# Sample 02: OpenAI SDK ഇന്റഗ്രേഷൻ

ഈ നോട്ട്‌ബുക്ക് OpenAI Python SDK-യുമായി ആധുനിക ഇന്റഗ്രേഷൻ പ്രദർശിപ്പിക്കുന്നു, Microsoft Foundry Local-നും Azure OpenAI-നും സ്ട്രീമിംഗ് പ്രതികരണങ്ങളോടും ശരിയായ പിശക് കൈകാര്യം ചെയ്യലോടും പിന്തുണ നൽകുന്നു.

## അവലോകനം

ഈ സാമ്പിൾ പ്രദർശിപ്പിക്കുന്നത്:
- Foundry Local-നും Azure OpenAI-നും ഇടയിൽ സുതാര്യമായ സ്വിച്ച് ചെയ്യൽ
- മെച്ചപ്പെട്ട ഉപയോക്തൃ അനുഭവത്തിനായി സ്ട്രീമിംഗ് ചാറ്റ് പൂർത്തീകരണങ്ങൾ
- FoundryLocalManager SDK-യുടെ ശരിയായ ഉപയോഗം
- ശക്തമായ പിശക് കൈകാര്യം ചെയ്യലും ഫാൾബാക്ക് മെക്കാനിസങ്ങളുമാണ്
- പ്രൊഡക്ഷൻ-സജ്ജമായ കോഡ് മാതൃകകൾ


## മുൻ‌വശം ആവശ്യങ്ങൾ

- **Foundry Local**: ഇൻസ്റ്റാൾ ചെയ്ത് പ്രവർത്തനക്ഷമമാക്കി (ലോകൽ ഇൻഫറൻസിനായി)
- **Python**: 3.8 അല്ലെങ്കിൽ അതിനുശേഷം OpenAI SDK-യോടുകൂടി
- **Azure OpenAI**: സാധുവായ എൻഡ്‌പോയിന്റും API കീയും (ക്ലൗഡ് ഇൻഫറൻസിനായി)

### ആശ്രിതങ്ങൾ ഇൻസ്റ്റാൾ ചെയ്യുക


In [ ]:
# Install required packages
!pip install openai foundry-local-sdk

## ലൈബ്രറികൾ ഇറക്കുമതി ചെയ്യുക ಮತ್ತು സജ്ജമാക്കുക


In [ ]:
import os
import sys
from openai import OpenAI
import time
from typing import Tuple

try:
    from foundry_local import FoundryLocalManager
    FOUNDRY_SDK_AVAILABLE = True
    print("✅ Foundry Local SDK is available")
except ImportError:
    FOUNDRY_SDK_AVAILABLE = False
    print("⚠️ Foundry Local SDK not available, manual configuration will be used")

## കോൺഫിഗറേഷൻ ഓപ്ഷനുകൾ

സംബന്ധിച്ച പരിസ്ഥിതി വ്യത്യാസങ്ങൾ സജ്ജമാക്കുന്നതിലൂടെ Azure OpenAI (ക്ലൗഡ്) അല്ലെങ്കിൽ Foundry Local (ഓൺ-ഡിവൈസ്) തിരഞ്ഞെടുക്കുക.


### ഓപ്ഷൻ 1: അസ്യൂർ ഓപ്പൺഎഐ കോൺഫിഗറേഷൻ

അൺകമ്മന്റ് ചെയ്ത് നിങ്ങളുടെ അസ്യൂർ ഓപ്പൺഎഐ ക്രെഡൻഷ്യലുകൾ സജ്ജമാക്കുക:


In [ ]:
# Azure OpenAI Configuration
# Uncomment and set your actual values

# os.environ["AZURE_OPENAI_ENDPOINT"] = "https://your-resource.openai.azure.com"
# os.environ["AZURE_OPENAI_API_KEY"] = "your-api-key-here"
# os.environ["AZURE_OPENAI_API_VERSION"] = "2024-08-01-preview"
# os.environ["MODEL"] = "your-deployment-name"  # e.g., "gpt-4"

print("Azure OpenAI configuration ready (if credentials are set)")

### ഓപ്ഷൻ 2: ഫൗണ്ട്രി ലോക്കൽ കോൺഫിഗറേഷൻ

ലോക്കൽ ഇൻഫറൻസിനുള്ള ഡിഫോൾട്ട് ക്രമീകരണങ്ങൾ:


In [ ]:
# Foundry Local Configuration (default)
FOUNDRY_MODEL = "phi-4-mini"  # Change to your preferred model
FOUNDRY_BASE_URL = "http://localhost:51211"
FOUNDRY_API_KEY = ""  # Usually empty for local

print(f"Foundry Local configuration ready with model: {FOUNDRY_MODEL}")

## ക്ലയന്റ് ഫാക്ടറി ഫംഗ്ഷനുകൾ

നിങ്ങളുടെ കോൺഫിഗറേഷനിന്റെ അടിസ്ഥാനത്തിൽ അനുയോജ്യമായ OpenAI ക്ലയന്റ് സൃഷ്ടിക്കുന്ന ഫംഗ്ഷനുകൾ ഇവയാണ്:


In [ ]:
def create_azure_client() -> Tuple[OpenAI, str]:
    """Create Azure OpenAI client."""
    azure_endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
    azure_api_key = os.environ.get("AZURE_OPENAI_API_KEY")
    azure_api_version = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
    
    if not azure_endpoint or not azure_api_key:
        raise ValueError("Azure OpenAI endpoint and API key are required")
    
    model = os.environ.get("MODEL", "your-deployment-name")
    client = OpenAI(
        base_url=f"{azure_endpoint}/openai",
        api_key=azure_api_key,
        default_query={"api-version": azure_api_version},
    )
    
    print(f"🌐 Azure OpenAI client created with model: {model}")
    return client, model


def create_foundry_client() -> Tuple[OpenAI, str]:
    """Create Foundry Local client with SDK management."""
    alias = FOUNDRY_MODEL
    
    if FOUNDRY_SDK_AVAILABLE:
        try:
            # Use FoundryLocalManager for proper service management
            print(f"🔄 Initializing Foundry Local with model: {alias}...")
            manager = FoundryLocalManager(alias)
            model_info = manager.get_model_info(alias)
            
            # Configure OpenAI client to use local Foundry service
            client = OpenAI(
                base_url=manager.endpoint,
                api_key=manager.api_key  # API key is not required for local usage
            )
            
            print(f"✅ Foundry Local SDK initialized")
            print(f"   Endpoint: {manager.endpoint}")
            print(f"   Model: {model_info.id}")
            return client, model_info.id
        except Exception as e:
            print(f"⚠️ Could not use Foundry SDK ({e}), falling back to manual configuration")
    
    # Fallback to manual configuration
    client = OpenAI(
        base_url=f"{FOUNDRY_BASE_URL}/v1",
        api_key=FOUNDRY_API_KEY
    )
    
    print(f"🔧 Manual Foundry Local configuration")
    print(f"   Endpoint: {FOUNDRY_BASE_URL}/v1")
    print(f"   Model: {alias}")
    return client, alias

## ക്ലയന്റ് ആരംഭിക്കുക

ഇത് സ്വയം കണ്ടെത്തുന്നു Azure OpenAI ഉപയോഗിക്കണോ അല്ലെങ്കിൽ Foundry Local:


In [ ]:
def initialize_client() -> Tuple[OpenAI, str, str]:
    """Initialize the appropriate OpenAI client."""
    
    # Check for Azure OpenAI configuration
    azure_endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
    azure_api_key = os.environ.get("AZURE_OPENAI_API_KEY")
    
    if azure_endpoint and azure_api_key:
        print("🌐 Azure OpenAI configuration detected")
        try:
            client, model = create_azure_client()
            return client, model, "azure"
        except Exception as e:
            print(f"❌ Azure OpenAI initialization failed: {e}")
            print("🔄 Falling back to Foundry Local...")
    
    # Use Foundry Local
    print("🏠 Using Foundry Local configuration")
    try:
        client, model = create_foundry_client()
        return client, model, "foundry"
    except Exception as e:
        print(f"❌ Foundry Local initialization failed: {e}")
        raise

# Initialize the client
print("Initializing OpenAI client...")
print("=" * 50)
client, model, provider = initialize_client()
print("=" * 50)
print(f"✅ Initialization complete! Using {provider} with model: {model}")

## അടിസ്ഥാന ചാറ്റ് പൂർത്തീകരണം

ഒരു ലളിതമായ ചാറ്റ് പൂർത്തീകരണം പരീക്ഷിക്കുക:


In [ ]:
def simple_chat(prompt: str, max_tokens: int = 150) -> str:
    """Send a simple chat message and get response."""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

# Test basic chat
test_prompt = "Say hello from the SDK quickstart and explain what you are in one sentence."

print(f"👤 User: {test_prompt}")
print("\n🤖 Assistant:")
response = simple_chat(test_prompt)
print(response)

## സ്റ്റ്രീമിംഗ് ചാറ്റ് പൂർത്തീകരണം

ഉപയോക്തൃ അനുഭവം മെച്ചപ്പെടുത്താൻ സ്റ്റ്രീമിംഗ് പ്രതികരണങ്ങൾ പ്രദർശിപ്പിക്കുക:


In [ ]:
def streaming_chat(prompt: str, max_tokens: int = 300) -> str:
    """Send a chat message with streaming response."""
    try:
        print("🤖 Assistant (streaming):")
        
        stream = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            stream=True
        )
        
        full_response = ""
        for chunk in stream:
            if chunk.choices[0].delta.content is not None:
                content = chunk.choices[0].delta.content
                print(content, end="", flush=True)
                full_response += content
        
        print("\n")  # New line after streaming
        return full_response
    except Exception as e:
        error_msg = f"Error: {e}"
        print(error_msg)
        return error_msg

# Test streaming chat
streaming_prompt = "Explain the key benefits of using Microsoft Foundry Local for AI development. Include aspects like privacy, performance, and cost."

print(f"👤 User: {streaming_prompt}\n")
streaming_response = streaming_chat(streaming_prompt)

## മൾട്ടി-ടേൺ സംഭാഷണം

സംഭാഷണത്തിന്റെ സാന്ദർഭ്യം നിലനിർത്തുന്നതിന്റെ ഉദാഹരണം:


In [ ]:
class ConversationManager:
    """Manages multi-turn conversations with context."""
    
    def __init__(self, system_prompt: str = None):
        self.messages = []
        if system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})
    
    def send_message(self, user_message: str, max_tokens: int = 200) -> str:
        """Send a message and get response while maintaining context."""
        # Add user message to conversation
        self.messages.append({"role": "user", "content": user_message})
        
        try:
            response = client.chat.completions.create(
                model=model,
                messages=self.messages,
                max_tokens=max_tokens
            )
            
            assistant_message = response.choices[0].message.content
            
            # Add assistant response to conversation
            self.messages.append({"role": "assistant", "content": assistant_message})
            
            return assistant_message
        except Exception as e:
            return f"Error: {e}"
    
    def get_conversation_length(self) -> int:
        """Get the number of messages in the conversation."""
        return len(self.messages)

# Create conversation manager with system prompt
system_prompt = "You are a helpful AI assistant specialized in explaining AI and machine learning concepts. Be concise but informative."
conversation = ConversationManager(system_prompt)

# Multi-turn conversation example
conversation_turns = [
    "What is the difference between AI inference on-device vs in the cloud?",
    "Which approach is better for privacy?",
    "What about performance and latency considerations?"
]

for i, turn in enumerate(conversation_turns, 1):
    print(f"\n{'='*60}")
    print(f"Turn {i}")
    print(f"{'='*60}")
    print(f"👤 User: {turn}")
    
    response = conversation.send_message(turn)
    print(f"\n🤖 Assistant: {response}")

print(f"\n📊 Conversation summary: {conversation.get_conversation_length()} messages total")

## പ്രകടന താരതമ്യം

വിവിധ സാഹചര്യങ്ങളിലെ പ്രതികരണ സമയങ്ങൾ താരതമ്യം ചെയ്യുക:


In [ ]:
def benchmark_response_time(prompt: str, iterations: int = 3) -> dict:
    """Benchmark response time for a given prompt."""
    times = []
    responses = []
    
    for i in range(iterations):
        start_time = time.time()
        
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=50  # Keep responses short for timing
            )
            
            end_time = time.time()
            response_time = end_time - start_time
            
            times.append(response_time)
            responses.append(response.choices[0].message.content)
            
        except Exception as e:
            print(f"Error in iteration {i+1}: {e}")
    
    if times:
        avg_time = sum(times) / len(times)
        min_time = min(times)
        max_time = max(times)
        
        return {
            "average_time": avg_time,
            "min_time": min_time,
            "max_time": max_time,
            "all_times": times,
            "sample_response": responses[0] if responses else None
        }
    
    return {"error": "No successful responses"}

# Benchmark different types of prompts
benchmark_prompts = [
    "What is AI?",
    "Explain machine learning in simple terms.",
    "List 3 benefits of edge computing."
]

print(f"⏱️  Performance Benchmark ({provider} - {model})")
print("=" * 60)

for prompt in benchmark_prompts:
    print(f"\n📝 Prompt: '{prompt}'")
    results = benchmark_response_time(prompt)
    
    if "error" not in results:
        print(f"   ⏰ Average time: {results['average_time']:.2f}s")
        print(f"   ⚡ Fastest: {results['min_time']:.2f}s")
        print(f"   🐌 Slowest: {results['max_time']:.2f}s")
        print(f"   📄 Sample response: {results['sample_response'][:100]}...")
    else:
        print(f"   ❌ {results['error']}")

## ആധുനിക കോൺഫിഗറേഷൻ ಮತ್ತು പിശക് കൈകാര്യം ചെയ്യൽ

വിവിധ പാരാമീറ്ററുകളും പിശക് സംഭവാവസ്ഥകളും പരീക്ഷിക്കുക:


In [ ]:
def test_different_parameters():
    """Test chat completions with different parameters."""
    prompt = "Write a creative short story about AI."
    
    # Test different temperature values
    temperatures = [0.1, 0.5, 0.9]
    
    for temp in temperatures:
        print(f"\n🌡️ Temperature: {temp}")
        print("-" * 30)
        
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=100,
                temperature=temp
            )
            
            print(f"Response: {response.choices[0].message.content[:150]}...")
            
        except Exception as e:
            print(f"Error with temperature {temp}: {e}")

test_different_parameters()

## സേവനാരോഗ്യ പരിശോധന

സമ്പൂർണ സേവനാരോഗ്യവും കഴിവും പരിശോധന:


In [ ]:
def comprehensive_health_check():
    """Perform comprehensive health check of the service."""
    print("🏥 Comprehensive Health Check")
    print("=" * 50)
    
    # 1. Check model listing
    try:
        models_response = client.models.list()
        available_models = [m.id for m in models_response.data]
        print(f"✅ Model listing: SUCCESS")
        print(f"   📋 Available models: {available_models}")
        
        if model in available_models:
            print(f"   ✅ Current model '{model}' is available")
        else:
            print(f"   ⚠️ Current model '{model}' not found in available models")
    except Exception as e:
        print(f"❌ Model listing: FAILED - {e}")
    
    # 2. Test basic completion
    try:
        test_response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Say 'Health check successful'"}],
            max_tokens=10
        )
        print(f"✅ Basic completion: SUCCESS")
        print(f"   💬 Response: {test_response.choices[0].message.content}")
    except Exception as e:
        print(f"❌ Basic completion: FAILED - {e}")
    
    # 3. Test streaming
    try:
        stream = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Count to 3"}],
            max_tokens=20,
            stream=True
        )
        
        stream_content = ""
        chunk_count = 0
        for chunk in stream:
            if chunk.choices[0].delta.content:
                stream_content += chunk.choices[0].delta.content
                chunk_count += 1
        
        print(f"✅ Streaming: SUCCESS")
        print(f"   📦 Chunks received: {chunk_count}")
        print(f"   💬 Streamed content: {stream_content.strip()}")
    except Exception as e:
        print(f"❌ Streaming: FAILED - {e}")
    
    # 4. Provider-specific information
    print(f"\n📊 Configuration Summary:")
    print(f"   🏢 Provider: {provider}")
    print(f"   🤖 Model: {model}")
    if provider == "foundry":
        print(f"   🏠 Foundry SDK Available: {FOUNDRY_SDK_AVAILABLE}")
        print(f"   🔗 Base URL: {FOUNDRY_BASE_URL}")
    elif provider == "azure":
        print(f"   🌐 Azure Endpoint: {os.environ.get('AZURE_OPENAI_ENDPOINT', 'Not set')}")
        print(f"   🔑 API Version: {os.environ.get('AZURE_OPENAI_API_VERSION', 'Not set')}")

comprehensive_health_check()

## ഇന്ററാക്ടീവ് ടെസ്റ്റിംഗ്

നിങ്ങളുടെ സ്വന്തം പ്രോംപ്റ്റുകൾ ഇന്ററാക്ടീവായി പരീക്ഷിക്കാൻ ഈ സെൽ ഉപയോഗിക്കുക:


In [ ]:
# Interactive testing - modify the prompt below
custom_prompt = "Explain the concept of 'edge AI' and why it's becoming important."
use_streaming = True  # Set to False for regular completion

print(f"👤 Custom Prompt: {custom_prompt}\n")

if use_streaming:
    custom_response = streaming_chat(custom_prompt, max_tokens=250)
else:
    custom_response = simple_chat(custom_prompt, max_tokens=250)
    print(f"🤖 Assistant: {custom_response}")

## സംഗ്രഹവും അടുത്ത ഘട്ടങ്ങളും

ഈ നോട്ട്‌ബുക്ക് താഴെപ്പറയുന്നവയുമായി ആധുനിക OpenAI SDK സംയോജനം പ്രദർശിപ്പിച്ചു:

### ✅ ഉൾപ്പെടുത്തിയ പ്രധാന സവിശേഷതകൾ

1. **മൾട്ടി-പ്രൊവൈഡർ പിന്തുണ**: Azure OpenAI-യും Foundry Local-ഉം തമ്മിൽ സുതാര്യമായ മാറൽ
2. **സ്റ്റ്രീമിംഗ് പ്രതികരണങ്ങൾ**: മെച്ചപ്പെട്ട ഉപയോക്തൃ അനുഭവത്തിനായി യഥാർത്ഥ സമയ ടോക്കൺ സൃഷ്ടി
3. **സംവാദം മാനേജ്മെന്റ്**: സാന്ദർഭ്യത്തോടെ മൾട്ടി-ടേൺ സംഭാഷണങ്ങൾ
4. **പ്രകടന ബഞ്ച്മാർക്കിംഗ്**: പ്രതികരണ സമയം അളക്കൽ, വിശകലനം
5. **സമഗ്ര ഹെൽത്ത് ചെക്കുകൾ**: സേവന പരിശോധനയും ഡയഗ്നോസ്റ്റിക്സും
6. **പിശക് കൈകാര്യം ചെയ്യൽ**: ശക്തമായ പിശക് കൈകാര്യം ചെയ്യലും ഫാൾബാക്ക് സംവിധാനങ്ങളും

### 🏆 Foundry Local vs Azure OpenAI

| വശം | Foundry Local | Azure OpenAI |
|--------|---------------|---------------|
| **സ്വകാര്യത** | ✅ ഡാറ്റ ലോക്കലിൽ തന്നെ നിലനിൽക്കുന്നു | ⚠️ ഡാറ്റ ക്ലൗഡിലേക്ക് അയയ്ക്കുന്നു |
| **വിലംബം** | ✅ കുറവ് (ലോകൽ ഇൻഫറൻസ്) | ⚠️ കൂടുതലാണ് (നെറ്റ്‌വർക്ക് ആശ്രിതം) |
| **ചെലവ്** | ✅ സൗജന്യം (ഹാർഡ്‌വെയർ കഴിഞ്ഞ്) | 💰 ടോക്കൺപ്രതി പണം നൽകണം |
| **ഓഫ്‌ലൈൻ** | ✅ ഓഫ്‌ലൈൻ പ്രവർത്തിക്കുന്നു | ❌ ഇന്റർനെറ്റ് ആവശ്യമാണ് |
| **മോഡൽ വൈവിധ്യം** | ⚠️ പരിമിതമായ തിരഞ്ഞെടുപ്പ് | ✅ പൂർണ്ണ മോഡൽ ആക്‌സസ് |
| **സ്കെയിലിംഗ്** | ⚠️ ഹാർഡ്‌വെയർ ആശ്രിതം | ✅ അനന്തമായ സ്കെയിലിംഗ് |

### 🚀 അടുത്ത ഘട്ടങ്ങൾ

- **സാമ്പിൾ 04**: Chainlit ചാറ്റ് അപ്ലിക്കേഷൻ നിർമ്മാണം
- **സാമ്പിൾ 05**: മൾട്ടി-ഏജന്റ് ഓർക്കസ്ട്രേഷൻ സിസ്റ്റങ്ങൾ
- **സാമ്പിൾ 06**: ബുദ്ധിമുട്ടുള്ള മോഡൽ റൂട്ടിംഗ്
- **പ്രൊഡക്ഷൻ ഡിപ്ലോയ്മെന്റ്**: സ്കെയിലിംഗ്, നിരീക്ഷണ പരിഗണനകൾ

### 💡 മികച്ച പ്രാക്ടീസുകൾ

1. **എപ്പോഴും പ്രൊവൈഡർമാരുടെ ഇടയിൽ ഫാൾബാക്ക് സംവിധാനങ്ങൾ നടപ്പിലാക്കുക**
2. **നീണ്ട പ്രതികരണങ്ങൾക്ക് സ്റ്റ്രീമിംഗ് ഉപയോഗിക്കുക** പ്രകടനമനുഭവം മെച്ചപ്പെടുത്താൻ
3. **പ്രൊഡക്ഷൻ അപ്ലിക്കേഷനുകൾക്കായി ശരിയായ പിശക് കൈകാര്യം ചെയ്യൽ നടപ്പിലാക്കുക**
4. **വിവിധ പ്രൊവൈഡർമാരുടെ പ്രതികരണ സമയം, ചെലവ് നിരീക്ഷിക്കുക**
5. **നിങ്ങളുടെ പ്രത്യേക ആവശ്യകതകൾ അടിസ്ഥാനമാക്കി ശരിയായ പ്രൊവൈഡർ തിരഞ്ഞെടുക്കുക**


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അസൂയാപത്രം**:  
ഈ രേഖ AI വിവർത്തന സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് വിവർത്തനം ചെയ്തതാണ്. നാം കൃത്യതയ്ക്ക് ശ്രമിച്ചിട്ടുണ്ടെങ്കിലും, സ്വയം പ്രവർത്തിക്കുന്ന വിവർത്തനങ്ങളിൽ പിശകുകൾ അല്ലെങ്കിൽ തെറ്റുകൾ ഉണ്ടാകാമെന്ന് ദയവായി ശ്രദ്ധിക്കുക. അതിന്റെ മാതൃഭാഷയിലുള്ള യഥാർത്ഥ രേഖയാണ് പ്രാമാണികമായ ഉറവിടം എന്ന് പരിഗണിക്കേണ്ടതാണ്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ വിവർത്തനം ശുപാർശ ചെയ്യപ്പെടുന്നു. ഈ വിവർത്തനം ഉപയോഗിക്കുന്നതിൽ നിന്നുണ്ടാകുന്ന ഏതെങ്കിലും തെറ്റിദ്ധാരണകൾക്കോ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കോ ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
